In [1]:
pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 3. Llama 3 Skill Extraction Function (The Comma-Separated Filter)
def extract_skills_with_ollama(resume_text):
    print("🧠 Sending raw resume to Llama 3 (Comma-Separated Mode)...")
    url = "http://localhost:11434/api/generate"
    
    # Notice we are no longer asking for JSON!
    prompt = f"""
    Extract all technical skills, programming languages, and ML frameworks from the text below.
    Return ONLY a single, comma-separated string of the skills.
    Do not use JSON. Do not use bullet points. Do not write sentences. 
    Example: Python, PyTorch, C++, SQL, NumPy, ResNet
    
    Text:
    {resume_text}
    """
    
    data = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False
        # Notice we removed "format": "json" completely!
    }
    
    try:
        response = requests.post(url, json=data)
        response_data = response.json()
        
        # Get the raw text string from Llama
        raw_text = response_data.get("response", "")
        
        # Clean it up: split by commas, strip away extra spaces, and make a clean Python list
        clean_skills = [skill.strip() for skill in raw_text.split(",") if skill.strip()]
        
        return clean_skills
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

In [3]:
import PyPDF2
import requests
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- Configuration ---
RESUME_FILE = "Harsh_resume.pdf"
OLLAMA_MODEL = "llama3" # Change this if you were using a different model name in Ollama
TARGET_JOB_ROLE = "Machine Learning Engineer / AI Researcher"
TARGET_REQUIRED_SKILLS = "Python PyTorch Deep Learning Computer Vision CNN Generative AI ResNet"

print("🚀 INITIALIZING ULTIMATE ATS ENGINE...")

# 1. Load the Sentence Transformer Model
print("⏳ Loading Embedding Model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. PDF Extraction Function
def extract_text_from_pdf(pdf_path):
    print(f"📄 Extracting text from {pdf_path}...")
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + " "
        return text.strip()
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{pdf_path}'")
        return None

# 3. Llama 3 Skill Extraction Function (The Filter)
def extract_skills_with_ollama(resume_text):
    print("🧠 Sending raw resume to Llama 3 to filter out the noise...")
    url = "http://localhost:11434/api/generate"
    
    # We strictly command Llama 3 to ONLY give us a JSON list
    prompt = f"""
    You are an expert technical recruiter. Extract all the technical skills, programming languages, 
    frameworks, and ML models from the resume text below. 
    Return ONLY a valid JSON list of strings. Do not include any other words, explanations, or formatting.
    Example output: ["Python", "PyTorch", "C++", "SQL"]
    
    Resume Text:
    {resume_text}
    """
    
    data = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json" # Forces Ollama to reply in strict JSON format
    }
    
    try:
        response = requests.post(url, json=data)
        response_data = response.json()
        skills_json_string = response_data.get("response", "[]")
        
        # Convert the string response into an actual Python list
        clean_skills_list = json.loads(skills_json_string)
        return clean_skills_list
    except Exception as e:
        print(f"❌ Llama 3 Error: {e}")
        return []

# --- MAIN EXECUTION PIPELINE ---
raw_text = extract_text_from_pdf(RESUME_FILE)

if raw_text:
    # Get the clean list of skills from Llama 3
    candidate_skills = extract_skills_with_ollama(raw_text)
    
    print("\n✨ Clean Skills Extracted by Llama 3:")
    print(candidate_skills)
    
    if candidate_skills:
        print("\n⚖️ Calculating Final Match Score...")
        
        # Turn the Llama 3 list back into a single string of pure keywords
        pure_skills_string = " ".join(candidate_skills)
        job_description = TARGET_JOB_ROLE + " " + TARGET_REQUIRED_SKILLS
        
        # Embed the pure skills and the job description
        jd_embedding = embedding_model.encode([job_description])
        resume_embedding = embedding_model.encode([pure_skills_string])
        
        # Calculate Similarity
        similarity = cosine_similarity(jd_embedding, resume_embedding)[0][0]
        final_score = round(float(similarity) * 100, 1)
        
        print("\n" + "="*50)
        print(f"🏆 HARSH RAJPUT'S TRUE ML ENGINEER SCORE: {final_score}%")
        print("="*50)
    else:
        print("⚠️ Llama 3 couldn't extract any skills. Make sure Ollama is running!")

C:\anaconda3\envs\ats_engine\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 INITIALIZING ULTIMATE ATS ENGINE...
⏳ Loading Embedding Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7572.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📄 Extracting text from Harsh_resume.pdf...
🧠 Sending raw resume to Llama 3 to filter out the noise...

✨ Clean Skills Extracted by Llama 3:
{'Python, PyTorch, C++, SQL, NumPy, Pandas, Matplotlib, Scikit-Learn, Regression, Clustering, Classification, ANN, CNN, RNN, NLP, U-Net, GPT-4, VGG-19, ResNet-50, ResNet18, DeepLabV3+': []}

⚖️ Calculating Final Match Score...

🏆 HARSH RAJPUT'S TRUE ML ENGINEER SCORE: 41.9%


In [4]:
import PyPDF2
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("🚀 INITIALIZING HYBRID ATS ENGINE...")

# 1. Configuration
RESUME_FILE = "Harsh_resume.pdf"
TARGET_JOB_ROLE = "Machine Learning Engineer / AI Researcher"

# We split the required skills into an actual list
REQUIRED_SKILLS = ["Python", "PyTorch", "Deep Learning", "Computer Vision", "CNN", "Generative AI", "ResNet"]
job_description = TARGET_JOB_ROLE + " " + " ".join(REQUIRED_SKILLS)

# 2. Load the AI
print("⏳ Loading Embedding Model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Read the PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + " "
        return text.strip()
    except:
        return ""

resume_text = extract_text_from_pdf(RESUME_FILE)

if resume_text:
    print("🧠 Analyzing Candidate Fit...")
    resume_text_lower = resume_text.lower()
    
    # --- METRIC 1: KEYWORD OVERLAP (The 'Resume Worded' Method) ---
    found_skills = []
    for skill in REQUIRED_SKILLS:
        # Check if the skill (lowercase) is anywhere in the resume text
        if skill.lower() in resume_text_lower:
            found_skills.append(skill)
            
    keyword_score = (len(found_skills) / len(REQUIRED_SKILLS)) * 100
    
    # --- METRIC 2: AI SEMANTIC SCORE (The Vector Method) ---
    jd_embedding = embedding_model.encode([job_description])
    resume_embedding = embedding_model.encode([resume_text])
    
    similarity = cosine_similarity(jd_embedding, resume_embedding)[0][0]
    # We normalize the AI score (Because 0.45 is technically a perfect match for massive documents)
    ai_score = min(float(similarity) * 200, 100.0) 
    
    # --- FINAL HYBRID CALCULATION ---
    # 60% weight to having the exact keywords, 40% weight to the AI's general context
    final_score = (keyword_score * 0.60) + (ai_score * 0.40)
    
    print("\n" + "="*50)
    print(f"🎯 TARGET ROLE: {TARGET_JOB_ROLE}")
    print(f"✅ REQUIRED SKILLS FOUND: {len(found_skills)}/{len(REQUIRED_SKILLS)} ({', '.join(found_skills)})")
    print("-" * 50)
    print(f"📊 Keyword Match Score: {round(keyword_score, 1)}%")
    print(f"🧠 AI Context Score:    {round(ai_score, 1)}%")
    print("=" * 50)
    print(f"🏆 HARSH RAJPUT'S FINAL ATS RANKING: {round(final_score, 1)}%")
    print("=" * 50)
else:
    print("❌ Could not read PDF.")

🚀 INITIALIZING HYBRID ATS ENGINE...
⏳ Loading Embedding Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7915.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Analyzing Candidate Fit...

🎯 TARGET ROLE: Machine Learning Engineer / AI Researcher
✅ REQUIRED SKILLS FOUND: 6/7 (Python, PyTorch, Deep Learning, CNN, Generative AI, ResNet)
--------------------------------------------------
📊 Keyword Match Score: 85.7%
🧠 AI Context Score:    91.6%
🏆 HARSH RAJPUT'S FINAL ATS RANKING: 88.1%


In [3]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

from gliner import GLiNER

# The model is already cached now, so it will load instantly!
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

raw_job_description = """
We are looking for a Machine Learning Engineer to join our AI Research team. 
The ideal candidate will have strong programming experience using Python and PyTorch. 
You should be familiar with Deep Learning principles, Computer Vision, and training CNNs. 
Hands-on experience with Generative AI workflows and optimizing ResNet architectures is a huge plus.
"""

# 1. HYPER-SPECIFIC LABELS
labels = ["programming language", "software framework", "machine learning model", "neural network architecture"]

print("🧠 Scanning Job Description for entities (Strict Mode)...")

# 2. ADD A CONFIDENCE THRESHOLD (Only accept entities the model is 70%+ sure about)
entities = model.predict_entities(raw_job_description, labels, threshold=0.7)

extracted_skills = []
for entity in entities:
    skill = entity["text"]
    if skill not in extracted_skills:
        extracted_skills.append(skill)

print("\n" + "="*50)
print("✨ CLEAN DYNAMIC SKILLS:")
print("="*50)
print(extracted_skills)

Fetching 5 files: 100%|██████████████████████████████████████████████████████████████████████████| 5/5 [00:00<?, ?it/s]


🧠 Scanning Job Description for entities (Strict Mode)...

✨ CLEAN DYNAMIC SKILLS:
['Python', 'PyTorch', 'CNNs', 'ResNet architectures']


In [4]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

import PyPDF2
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gliner import GLiNER

print("🚀 INITIALIZING FULLY DYNAMIC AI ATS ENGINE...")

# 1. Configuration (No more hardcoded skills!)
RESUME_FILE = "Harsh_resume.pdf"
TARGET_JOB_ROLE = "Machine Learning Engineer"

# You can paste ANY raw job description from LinkedIn here now!
RAW_JOB_DESCRIPTION = """
We are looking for a Machine Learning Engineer to join our AI Research team. 
The ideal candidate will have strong programming experience using Python and PyTorch. 
You should be familiar with Deep Learning principles, Computer Vision, and training CNNs. 
Hands-on experience with Generative AI workflows and optimizing ResNet architectures is a huge plus.
"""

# 2. Load the AI Models
print("⏳ Loading GLiNER (Skill Extractor) & Sentence Transformer (Semantic Scorer)...")
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Dynamically Extract Required Skills
print("🧠 Extracting target skills from Job Description...")
labels = ["programming language", "software framework", "machine learning model", "neural network architecture"]
entities = gliner_model.predict_entities(RAW_JOB_DESCRIPTION, labels, threshold=0.7)

REQUIRED_SKILLS = []
for entity in entities:
    skill = entity["text"]
    if skill not in REQUIRED_SKILLS:
        REQUIRED_SKILLS.append(skill)

# 4. Read the PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + " "
        return text.strip()
    except Exception as e:
        print(f"❌ PDF Error: {e}")
        return ""

resume_text = extract_text_from_pdf(RESUME_FILE)

if resume_text:
    print("⚖️ Calculating Candidate Fit...")
    resume_text_lower = resume_text.lower()
    
    # --- METRIC 1: KEYWORD OVERLAP ---
    found_skills = []
    for skill in REQUIRED_SKILLS:
        # Check if the extracted skill is in the resume
        if skill.lower() in resume_text_lower:
            found_skills.append(skill)
            
    # Protect against division by zero if GLiNER finds nothing
    keyword_score = (len(found_skills) / len(REQUIRED_SKILLS)) * 100 if REQUIRED_SKILLS else 0
    
    # --- METRIC 2: AI SEMANTIC SCORE ---
    # We compare the entire resume against the raw job description
    jd_embedding = embedding_model.encode([TARGET_JOB_ROLE + " " + RAW_JOB_DESCRIPTION])
    resume_embedding = embedding_model.encode([resume_text])
    
    similarity = cosine_similarity(jd_embedding, resume_embedding)[0][0]
    ai_score = min(float(similarity) * 200, 100.0) 
    
    # --- FINAL HYBRID CALCULATION ---
    final_score = (keyword_score * 0.60) + (ai_score * 0.40)
    
    print("\n" + "="*50)
    print(f"🎯 TARGET ROLE: {TARGET_JOB_ROLE}")
    print(f"📌 DYNAMIC SKILLS DETECTED: {', '.join(REQUIRED_SKILLS)}")
    print(f"✅ SKILLS MATCHED: {len(found_skills)}/{len(REQUIRED_SKILLS)} ({', '.join(found_skills)})")
    print("-" * 50)
    print(f"📊 Keyword Match Score: {round(keyword_score, 1)}%")
    print(f"🧠 AI Context Score:    {round(ai_score, 1)}%")
    print("=" * 50)
    print(f"🏆 HARSH RAJPUT'S FINAL ATS RANKING: {round(final_score, 1)}%")
    print("=" * 50)
else:
    print("❌ Could not read PDF.")

🚀 INITIALIZING FULLY DYNAMIC AI ATS ENGINE...
⏳ Loading GLiNER (Skill Extractor) & Sentence Transformer (Semantic Scorer)...


C:\anaconda3\envs\ats_engine\lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 509.86it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Extracting target skills from Job Description...
⚖️ Calculating Candidate Fit...

🎯 TARGET ROLE: Machine Learning Engineer
📌 DYNAMIC SKILLS DETECTED: Python, PyTorch, CNNs, ResNet architectures
✅ SKILLS MATCHED: 2/4 (Python, PyTorch)
--------------------------------------------------
📊 Keyword Match Score: 50.0%
🧠 AI Context Score:    73.5%
🏆 HARSH RAJPUT'S FINAL ATS RANKING: 59.4%
